# Dependency Injection & Enterprise Patterns

## What's covered

- **Dependency Injection & Inversion of Control** — what DI actually means, the three injection styles, and when an IoC container earns its complexity
- **Repository** — encapsulate data access behind a collection-like interface
- **Unit of Work** — track changes across a business transaction and commit them atomically
- **Specification** — encode business rules as composable predicates
- **Null Object** — replace `null` with a neutral object that supports the same interface
- How these patterns combine in a typical service layer


## The shape of modern service-layer code

The GoF catalog was written before web services, ORMs, and dependency-injection frameworks were standard. The patterns in this notebook are the ones that grew up around modern service-oriented codebases. Most of them are *applications* of the GoF patterns (Repository is a Facade over data access; Specification is a Strategy with composition baked in), framed for the specific structure of a service-and-domain layered application.

A typical service-layer flow:

1. A **request** arrives at a controller (HTTP, RPC, message queue).
2. The controller calls a **service**, passing the request payload.
3. The service uses one or more **repositories** to load entities and persists changes through a **unit of work**.
4. Business rules are expressed as **specifications** — predicates that can be combined.
5. Anywhere a "no value yet" or "default behaviour" would force a `null` check, a **null object** provides a no-op stand-in.

The whole layer is wired with **dependency injection** — each service receives its repositories, its unit of work, and its specifications through its constructor, so tests can substitute fakes for any of them.


## Dependency Injection & Inversion of Control

**Inversion of Control (IoC)** is a broad principle: the framework or the higher-level component decides *when* your code runs and *what* it gets handed. **Dependency Injection (DI)** is a specific technique for implementing IoC at the object level — components receive their dependencies from outside instead of constructing them.

The principle is the **Dependency Inversion Principle** from notebook 01 — depend on abstractions, not concretions. DI is the *how*: instead of a `UserService` constructing its own `PostgresUserRepo`, the service accepts a `UserRepo` in its constructor and the wiring code at the top of the application chooses which concrete repo to pass in.

**Three injection styles:**

- **Constructor injection** — pass dependencies as constructor arguments. The default, the safest, and the most testable. A constructed object is fully ready to use; required dependencies cannot be missing.
- **Setter injection** — dependencies are assigned to fields after construction. Useful when a dependency is optional or when the framework needs to break a circular construction order.
- **Method injection** — dependencies are passed as parameters to specific methods. Right when only one method needs the dependency and storing it on the object would be misleading.

**Do you need an IoC container?** A *container* is a framework (Spring, Guice, Dagger, Koin, dependency-injector, etc.) that constructs your object graph automatically based on type declarations or configuration. Containers shine in large applications where the wiring would otherwise be hundreds of lines of construction code. In small to medium codebases, **manual DI** — wiring up dependencies by hand in your `main()` function or composition root — is usually clearer.


### Python


In [ ]:
from typing import Protocol


# Abstractions (the "ports" in hexagonal architecture).
class EmailSender(Protocol):
    def send(self, to: str, subject: str, body: str) -> None: ...


class UserRepo(Protocol):
    def find(self, user_id: int) -> dict | None: ...


# Concrete implementations.
class SmtpEmailSender:
    def __init__(self, host: str):
        self.host = host

    def send(self, to: str, subject: str, body: str) -> None:
        print(f"  [smtp:{self.host}] to {to}: {subject}")


class InMemoryUserRepo:
    def __init__(self, users: dict[int, dict]):
        self.users = users

    def find(self, user_id: int) -> dict | None:
        return self.users.get(user_id)


# Service — depends on abstractions, not concretions.
class WelcomeService:
    # Constructor injection — required dependencies arrive ready to use.
    def __init__(self, repo: UserRepo, email: EmailSender):
        self._repo = repo
        self._email = email

    def send_welcome(self, user_id: int) -> None:
        user = self._repo.find(user_id)
        if user is None:
            raise LookupError(f"no user {user_id}")
        self._email.send(user["email"], "Welcome!", f"Hi {user['name']}, glad you joined.")


# Composition root — the one place that knows about concrete classes.
def main() -> None:
    repo = InMemoryUserRepo({1: {"id": 1, "name": "Alice", "email": "alice@example.com"}})
    email = SmtpEmailSender(host="smtp.example.com")
    service = WelcomeService(repo=repo, email=email)   # explicit wiring
    service.send_welcome(1)


# In tests we can wire fakes in.
class FakeEmail:
    def __init__(self):
        self.sent: list[tuple[str, str, str]] = []
    def send(self, to: str, subject: str, body: str) -> None:
        self.sent.append((to, subject, body))


def test_send_welcome() -> None:
    repo = InMemoryUserRepo({1: {"id": 1, "name": "A", "email": "a@x"}})
    fake = FakeEmail()
    WelcomeService(repo=repo, email=fake).send_welcome(1)
    assert fake.sent == [("a@x", "Welcome!", "Hi A, glad you joined.")]
    print("  test passed")


main()
test_send_welcome()


### Kotlin

```kotlin
// Abstractions
interface EmailSender { fun send(to: String, subject: String, body: String) }
interface UserRepo { fun find(userId: Int): User? }
data class User(val id: Int, val name: String, val email: String)

// Concrete implementations
class SmtpEmailSender(private val host: String) : EmailSender {
    override fun send(to: String, subject: String, body: String) {
        println("  [smtp:$host] to $to: $subject")
    }
}

class InMemoryUserRepo(private val users: Map<Int, User>) : UserRepo {
    override fun find(userId: Int) = users[userId]
}

// Service with constructor injection
class WelcomeService(private val repo: UserRepo, private val email: EmailSender) {
    fun sendWelcome(userId: Int) {
        val user = repo.find(userId) ?: error("no user $userId")
        email.send(user.email, "Welcome!", "Hi ${user.name}, glad you joined.")
    }
}

// Composition root
fun main() {
    val repo = InMemoryUserRepo(mapOf(1 to User(1, "Alice", "alice@example.com")))
    val email = SmtpEmailSender("smtp.example.com")
    WelcomeService(repo, email).sendWelcome(1)
}

main()
```


**When you don't need a DI container.** Small to medium projects with a clear composition root can be wired by hand in 30 lines of `main()`. Containers earn their complexity at scale — hundreds of services, scoped lifecycles (singleton vs. request-scoped vs. transient), AOP cross-cuts. Don't introduce a container reflexively because "real codebases have one"; introduce one when the manual wiring is genuinely the bottleneck.


## Repository

**Intent:** mediate between the domain and data-mapping layers using a collection-like interface for accessing domain objects.

**Where it shows up:** every modern service-layer codebase. The repository hides whatever sits underneath — relational database, document store, REST API, in-memory dictionary — and exposes methods that read and write *domain objects*. Business logic calls `users.find_by_email(...)`; whether that becomes a SQL query, a GET request, or a dict lookup is the repository's problem.

**The shape:** an interface like `UserRepo` with methods such as `find(id)`, `find_by_email(email)`, `add(user)`, `remove(user)`. Concrete classes (`PostgresUserRepo`, `RestUserRepo`, `InMemoryUserRepo`) implement the interface. Business logic depends on the interface.

**Why this matters:**

- **Testability.** Swap an in-memory implementation in for tests; no database needed.
- **Migration safety.** Switching from Postgres to DynamoDB touches only the repository; business logic doesn't change.
- **Vocabulary.** The repository speaks in domain terms (`find_active_subscribers()`) rather than SQL (`SELECT * FROM users WHERE status = 'active' ...`).

**Repository versus DAO (Data Access Object):** DAO is generic CRUD plumbing per table. Repository is *domain-shaped* — the methods correspond to use cases, not to tables. `OrderRepo.find_pending_for_warehouse(wh)` is a repository method. `OrderDao.find_by_status(status)` is a DAO method.


### Python


In [ ]:
from dataclasses import dataclass
from typing import Protocol


@dataclass
class Order:
    id: int
    customer: str
    total: float
    status: str   # "pending", "paid", "shipped"


class OrderRepo(Protocol):
    def find(self, order_id: int) -> Order | None: ...
    def find_pending_for_customer(self, customer: str) -> list[Order]: ...
    def add(self, order: Order) -> None: ...
    def update(self, order: Order) -> None: ...


# In-memory implementation — perfect for tests and small apps.
class InMemoryOrderRepo:
    def __init__(self, seed: list[Order] | None = None):
        self._by_id: dict[int, Order] = {o.id: o for o in (seed or [])}

    def find(self, order_id: int) -> Order | None:
        return self._by_id.get(order_id)

    def find_pending_for_customer(self, customer: str) -> list[Order]:
        return [o for o in self._by_id.values()
                if o.customer == customer and o.status == "pending"]

    def add(self, order: Order) -> None:
        self._by_id[order.id] = order

    def update(self, order: Order) -> None:
        if order.id not in self._by_id:
            raise LookupError(f"order {order.id} not found")
        self._by_id[order.id] = order


# A "Postgres" implementation — same interface, different innards.
class PostgresOrderRepo:
    def __init__(self, connection):
        self.conn = connection

    def find(self, order_id: int) -> Order | None:
        # imagine: SELECT id, customer, total, status FROM orders WHERE id = %s
        return None  # stub

    def find_pending_for_customer(self, customer: str) -> list[Order]:
        # imagine: SELECT ... WHERE customer = %s AND status = 'pending'
        return []  # stub

    def add(self, order: Order) -> None:
        pass  # stub: INSERT INTO orders ...

    def update(self, order: Order) -> None:
        pass  # stub: UPDATE orders SET ... WHERE id = %s


# Business logic depends on the protocol; concrete repo arrives via DI.
def total_pending(repo: OrderRepo, customer: str) -> float:
    return sum(o.total for o in repo.find_pending_for_customer(customer))


repo = InMemoryOrderRepo([
    Order(1, "alice", 30.0, "pending"),
    Order(2, "alice", 45.0, "paid"),
    Order(3, "alice", 25.0, "pending"),
    Order(4, "bob",   99.0, "pending"),
])
print(total_pending(repo, "alice"))   # 55.0
print(total_pending(repo, "bob"))     # 99.0


### Kotlin

```kotlin
data class Order(val id: Int, val customer: String, val total: Double, val status: String)

interface OrderRepo {
    fun find(orderId: Int): Order?
    fun findPendingForCustomer(customer: String): List<Order>
    fun add(order: Order)
    fun update(order: Order)
}

class InMemoryOrderRepo(seed: List<Order> = emptyList()) : OrderRepo {
    private val byId = seed.associateBy { it.id }.toMutableMap()

    override fun find(orderId: Int) = byId[orderId]
    override fun findPendingForCustomer(customer: String) =
        byId.values.filter { it.customer == customer && it.status == "pending" }
    override fun add(order: Order) { byId[order.id] = order }
    override fun update(order: Order) {
        require(byId.containsKey(order.id)) { "order ${order.id} not found" }
        byId[order.id] = order
    }
}

fun totalPending(repo: OrderRepo, customer: String): Double =
    repo.findPendingForCustomer(customer).sumOf { it.total }

val repo = InMemoryOrderRepo(listOf(
    Order(1, "alice", 30.0, "pending"),
    Order(2, "alice", 45.0, "paid"),
    Order(3, "alice", 25.0, "pending"),
    Order(4, "bob", 99.0, "pending"),
))
println(totalPending(repo, "alice"))
println(totalPending(repo, "bob"))
```


**When not to use Repository.** When the application is small enough that the database is the model — a CRUD admin tool or a script. The repository's overhead (interface, two implementations, dependency wiring) is justified when you have actual business logic that should not know about persistence, and when tests would benefit from swapping in an in-memory fake.


## Unit of Work

**Intent:** maintain a list of objects affected by a business transaction and coordinate writing out the changes and resolving concurrency problems.

**Where it shows up:** every ORM has one — SQLAlchemy's `Session`, Hibernate's `Session`, Entity Framework's `DbContext`, Doctrine's `EntityManager`. The unit of work tracks which entities were loaded, which were modified, which are new, and which are deleted. On `commit()`, it writes all the changes in one database transaction.

**Why the pattern exists:** without it, business logic would have to remember to `save()` every modified entity explicitly, ordering writes correctly, handling transaction boundaries by hand. The Unit of Work centralizes that bookkeeping.

**The shape:** a unit-of-work object with methods to register new entities, mark dirty ones, mark deleted ones, plus `commit()` and `rollback()`. Repositories live inside the unit of work and feed it changes. Business logic talks to the repositories; the unit of work synthesizes the writes.

**Pairing with Repository:** repositories handle reads and tell the unit of work about writes. The two patterns are designed to work together — Repository is the read/write surface, Unit of Work is the transaction boundary.


### Python


In [ ]:
from dataclasses import dataclass, field
from typing import Protocol


@dataclass
class Product:
    id: int
    name: str
    stock: int


class UnitOfWork(Protocol):
    products: "ProductRepository"
    def commit(self) -> None: ...
    def rollback(self) -> None: ...


class ProductRepository:
    def __init__(self, uow: "InMemoryUnitOfWork"):
        self._uow = uow

    def get(self, product_id: int) -> Product | None:
        # mark as "seen" so we can detect dirty updates on commit
        product = self._uow._store.get(product_id)
        if product is not None:
            self._uow._seen[product_id] = Product(**product.__dict__)  # snapshot
        return product

    def add(self, product: Product) -> None:
        self._uow._new.append(product)


class InMemoryUnitOfWork:
    def __init__(self, seed: list[Product] | None = None):
        self._store: dict[int, Product] = {p.id: p for p in (seed or [])}
        self._seen: dict[int, Product] = {}
        self._new: list[Product] = []
        self.products = ProductRepository(self)

    def commit(self) -> None:
        # apply new entities
        for p in self._new:
            self._store[p.id] = p
        # detect and apply dirty entities (those whose live values differ from the snapshot)
        for pid, snapshot in self._seen.items():
            live = self._store.get(pid)
            if live and live != snapshot:
                print(f"  [commit] updated {pid}: stock {snapshot.stock} -> {live.stock}")
        print(f"  [commit] added {len(self._new)} new")
        self._new.clear()
        self._seen.clear()

    def rollback(self) -> None:
        # restore snapshots
        for pid, snapshot in self._seen.items():
            self._store[pid] = snapshot
        self._new.clear()
        self._seen.clear()
        print("  [rollback]")

    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc, tb):
        if exc_type is None:
            self.commit()
        else:
            self.rollback()


# Business use case: fulfill an order — atomic across multiple product updates.
def fulfill_order(uow: InMemoryUnitOfWork, items: dict[int, int]) -> None:
    with uow:
        for product_id, quantity in items.items():
            product = uow.products.get(product_id)
            if product is None or product.stock < quantity:
                raise ValueError(f"cannot fulfill {product_id}")
            product.stock -= quantity


uow = InMemoryUnitOfWork([
    Product(1, "widget", 100),
    Product(2, "gizmo",   50),
])
fulfill_order(uow, {1: 5, 2: 3})
print(uow._store)

# rollback on failure
try:
    fulfill_order(uow, {1: 5, 2: 999})
except ValueError as e:
    print(f"  failed: {e}")
print(uow._store)   # unchanged from the previous commit


### Kotlin

```kotlin
data class Product(val id: Int, val name: String, var stock: Int)

class ProductRepository(private val uow: InMemoryUnitOfWork) {
    fun get(id: Int): Product? {
        val p = uow.store[id]
        if (p != null && id !in uow.seen) uow.seen[id] = p.copy()  // snapshot
        return p
    }
    fun add(p: Product) { uow.newEntities.add(p) }
}

class InMemoryUnitOfWork(seed: List<Product> = emptyList()) {
    val store = seed.associateBy { it.id }.toMutableMap()
    val seen = mutableMapOf<Int, Product>()
    val newEntities = mutableListOf<Product>()
    val products = ProductRepository(this)

    fun commit() {
        newEntities.forEach { store[it.id] = it }
        seen.forEach { (id, snap) ->
            val live = store[id]
            if (live != null && live != snap) {
                println("  [commit] updated $id: stock ${snap.stock} -> ${live.stock}")
            }
        }
        println("  [commit] added ${newEntities.size} new")
        newEntities.clear(); seen.clear()
    }

    fun rollback() {
        seen.forEach { (id, snap) -> store[id] = snap }
        newEntities.clear(); seen.clear()
        println("  [rollback]")
    }
}

inline fun <T> InMemoryUnitOfWork.transaction(block: (InMemoryUnitOfWork) -> T): T = try {
    val result = block(this); commit(); result
} catch (e: Throwable) { rollback(); throw e }

fun fulfillOrder(uow: InMemoryUnitOfWork, items: Map<Int, Int>) {
    uow.transaction {
        items.forEach { (pid, qty) ->
            val p = uow.products.get(pid) ?: error("missing $pid")
            require(p.stock >= qty) { "insufficient $pid" }
            p.stock -= qty
        }
    }
}
```


**When not to use Unit of Work.** When you're not crossing a transaction boundary — a simple read-only query or a single-row write doesn't need it. Also when you're already using an ORM that provides a Unit of Work for you (SQLAlchemy session, etc.) — wrapping it in another layer is duplication. Reach for an explicit Unit of Work when business operations span multiple repositories and must be atomic, especially when you want the option to test with an in-memory implementation.


## Specification

**Intent:** encapsulate business rules as predicates that can be combined and inspected.

**Where it shows up:** complex filter logic ("eligible customers" = active AND (premium OR employee) AND NOT cancelled), validation rules, query building, authorization checks.

**The shape:** each rule is a small object (or callable) with a single method that returns true/false for a candidate. Specifications support `and`, `or`, `not` combinators, so you can build complex rules from simple parts.

**Why not just write `if x.active and (x.premium or x.is_employee())`?** Three reasons:

- **Reuse.** The same predicate is used in filtering, validation, and authorization. Defining it once means changing it once.
- **Naming.** `is_eligible_for_discount` carries domain meaning. A multi-clause boolean expression does not.
- **Inspection.** A specification *object* can be introspected — translated to SQL, rendered for an admin UI, serialized for audit. A raw lambda cannot.

**Python's take:** a `Specification` protocol with `is_satisfied_by()`, plus combinator methods. Or — for simpler cases — just compose callables.

**Kotlin's take:** the same with interfaces; for simple cases, functions returning booleans, combined with infix `and`/`or` functions.


### Python


In [ ]:
from abc import ABC, abstractmethod
from dataclasses import dataclass


@dataclass
class Customer:
    id: int
    name: str
    active: bool
    premium: bool
    employee: bool
    cancelled: bool


class Spec(ABC):
    @abstractmethod
    def is_satisfied_by(self, c: Customer) -> bool: ...

    def __and__(self, other: "Spec") -> "Spec": return And(self, other)
    def __or__(self, other: "Spec") -> "Spec":  return Or(self, other)
    def __invert__(self) -> "Spec":             return Not(self)


@dataclass
class And(Spec):
    a: Spec
    b: Spec
    def is_satisfied_by(self, c: Customer) -> bool:
        return self.a.is_satisfied_by(c) and self.b.is_satisfied_by(c)


@dataclass
class Or(Spec):
    a: Spec
    b: Spec
    def is_satisfied_by(self, c: Customer) -> bool:
        return self.a.is_satisfied_by(c) or self.b.is_satisfied_by(c)


@dataclass
class Not(Spec):
    a: Spec
    def is_satisfied_by(self, c: Customer) -> bool:
        return not self.a.is_satisfied_by(c)


# Leaf specifications — domain-named, single-purpose.
class IsActive(Spec):
    def is_satisfied_by(self, c: Customer) -> bool: return c.active


class IsPremium(Spec):
    def is_satisfied_by(self, c: Customer) -> bool: return c.premium


class IsEmployee(Spec):
    def is_satisfied_by(self, c: Customer) -> bool: return c.employee


class IsCancelled(Spec):
    def is_satisfied_by(self, c: Customer) -> bool: return c.cancelled


# Combine into a high-level domain rule.
eligible_for_discount = IsActive() & (IsPremium() | IsEmployee()) & ~IsCancelled()


customers = [
    Customer(1, "alice", active=True, premium=True, employee=False, cancelled=False),
    Customer(2, "bob",   active=True, premium=False, employee=True, cancelled=False),
    Customer(3, "carol", active=False, premium=True, employee=False, cancelled=False),
    Customer(4, "dave",  active=True, premium=True, employee=False, cancelled=True),
]

for c in customers:
    print(f"  {c.name}: {eligible_for_discount.is_satisfied_by(c)}")


### Kotlin

```kotlin
data class Customer(
    val id: Int, val name: String,
    val active: Boolean, val premium: Boolean,
    val employee: Boolean, val cancelled: Boolean,
)

fun interface Spec { fun isSatisfiedBy(c: Customer): Boolean }

infix fun Spec.and(other: Spec) = Spec { c -> isSatisfiedBy(c) && other.isSatisfiedBy(c) }
infix fun Spec.or(other: Spec)  = Spec { c -> isSatisfiedBy(c) || other.isSatisfiedBy(c) }
operator fun Spec.not()         = Spec { c -> !isSatisfiedBy(c) }

val isActive    = Spec { it.active }
val isPremium   = Spec { it.premium }
val isEmployee  = Spec { it.employee }
val isCancelled = Spec { it.cancelled }

val eligibleForDiscount = isActive and (isPremium or isEmployee) and !isCancelled

val customers = listOf(
    Customer(1, "alice", true, true, false, false),
    Customer(2, "bob", true, false, true, false),
    Customer(3, "carol", false, true, false, false),
    Customer(4, "dave", true, true, false, true),
)

customers.forEach { println("  ${it.name}: ${eligibleForDiscount.isSatisfiedBy(it)}") }
```


**When not to use Specification.** When the rule is one-line and appears in exactly one place — just write the `if`. Specification earns its keep when the rule is reused across filtering, validation, and authorization; when domain names matter for readability; or when you need to translate the rule into a different form (a SQL `WHERE` clause, a UI badge, an audit log entry).


## Null Object

**Intent:** provide an object that exposes the expected interface but does nothing useful, as a stand-in for the "no value" case. The caller treats it like any other instance — no null checks required.

**Where it shows up:** a `GuestUser` for unauthenticated requests (has the same interface as `RegisteredUser` but returns sensible defaults). A `NoOpLogger` that swallows messages. An `EmptyCart` that always shows zero items. A `NullObserver` that ignores every event.

**The shape:** the null object implements the same interface as the real one. Every method is a no-op or returns a neutral default. Callers can substitute it for the real thing without changing their code.

**Python's take:** a class with methods that return sensible defaults. Sometimes a module-level constant (`NULL_USER = GuestUser()`) so callers can use `is`-comparison if needed.

**Kotlin's take:** the same. Kotlin's nullable types (`User?`) often substitute for Null Object at the type level — but Null Object still shines when the *behaviour* of the null case is non-trivial (a guest user has different permissions, not no permissions).

**Null Object versus nullable types:** the two solve different problems. Nullable types make absence *explicit at the type level* — callers must handle `null` or the type checker complains. Null Object makes absence *behaviourally invisible* — callers never see it. Use nullable types when "no value" needs different handling. Use Null Object when "no value" maps onto a useful default behaviour.


### Python


In [ ]:
from typing import Protocol


class User(Protocol):
    name: str
    def can(self, action: str) -> bool: ...


class RegisteredUser:
    def __init__(self, name: str, permissions: set[str]):
        self.name = name
        self._permissions = permissions

    def can(self, action: str) -> bool:
        return action in self._permissions


# Null object — same interface, sensible defaults.
class GuestUser:
    name = "guest"
    def can(self, action: str) -> bool:
        return action == "view_public"


GUEST = GuestUser()  # singleton — there's only one notion of "guest"


def current_user(session_id: str | None) -> User:
    if session_id is None:
        return GUEST
    # imagine a real lookup
    return RegisteredUser("alice", permissions={"view_public", "edit_post", "delete_post"})


# Caller code does not change shape between guest and registered.
def render_navbar(user: User) -> str:
    parts = [f"Hello, {user.name}"]
    if user.can("edit_post"):
        parts.append("[New post]")
    if user.can("delete_post"):
        parts.append("[Admin]")
    return " | ".join(parts)


print(render_navbar(current_user(session_id=None)))      # guest path
print(render_navbar(current_user(session_id="sess123"))) # registered path


### Kotlin

```kotlin
interface User {
    val name: String
    fun can(action: String): Boolean
}

class RegisteredUser(override val name: String, private val permissions: Set<String>) : User {
    override fun can(action: String) = action in permissions
}

object GuestUser : User {
    override val name = "guest"
    override fun can(action: String) = action == "view_public"
}

fun currentUser(sessionId: String?): User =
    if (sessionId == null) GuestUser
    else RegisteredUser("alice", setOf("view_public", "edit_post", "delete_post"))

fun renderNavbar(user: User): String = buildList {
    add("Hello, ${user.name}")
    if (user.can("edit_post"))   add("[New post]")
    if (user.can("delete_post")) add("[Admin]")
}.joinToString(" | ")

println(renderNavbar(currentUser(null)))
println(renderNavbar(currentUser("sess123")))
```


**When not to use Null Object.** When the "no value" case really needs different handling at the call site — make absence visible with a nullable type or an `Optional`. Null Object hides the absence; if every caller has to behave differently, hiding it just pushes the special case into the null object's methods, which then has to know about every caller's needs. Use Null Object when the default behaviour is *genuinely shared* across callers.


## How these patterns combine

A typical service-layer method, wiring all five together:

1. The application's composition root constructs the unit of work, all repositories, and any specifications. (**Dependency Injection**)
2. A controller calls `service.fulfill_order(customer_id, items)`.
3. The service uses the unit of work to load the customer through `users.find(customer_id)`. (**Repository**)
4. If the customer is not found, the service raises — *not* a null. (Or it could use a **Null Object** like `GuestCustomer` if "no customer" maps to a valid default.)
5. The service checks `eligible_for_discount.is_satisfied_by(customer)` to apply pricing rules. (**Specification**)
6. The service modifies the customer and the order through their repositories.
7. On success, the unit of work commits all writes in one transaction. (**Unit of Work**)

Every dependency in this flow is constructor-injected, so the whole service layer can be tested with in-memory repositories, an in-memory unit of work, and the same real specifications. No database. No mocks of the service itself. The patterns reinforce each other.


Notebook seven closes the series with **Anti-Patterns & Pattern Selection**. The common ways patterns fail in practice — god classes, singleton abuse, anemic domain models, pattern fever — and a decision guide for choosing patterns based on the smell that's actually appeared, rather than on speculation.
